In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download the `.pkg` and install it
   - Windows: download the `.msi` and install it
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```


In [5]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I don’t see any FAQ entry about **running Ollama locally** in the provided context.

The closest related guidance is that you can run the course locally if you’re comfortable setting up the needed tools yourself, including Python, `uv`, Jupyter, Docker, and any other module-specific tools, and you should document your setup to keep it reproducible.

If you want, I can also help you figure out a local setup based on that guidance.


In [6]:
messages = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

In [7]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
)

print(response.output_text)

It depends on the course’s enrollment rules.

- **If it’s open enrollment:** you should be able to join immediately from the course page (often with a **Join/Enroll** button).
- **If it has prerequisites or a start date:** you may need to meet requirements or wait until the next enrollment window.
- **If it’s invite-only:** you’ll need an invite or admin approval.

If you tell me the **course platform/name** (or paste the course link and what it says on the page), I can tell you the exact way to check and join.


In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query, num_results=5, boost_dict=boost_dict, filter_dict=filter_dict
    )

In [9]:
search("How do I find Ollama?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [12]:
len(response.output)

1

In [13]:
call = response.output[0]

In [14]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join admission late enrollment eligibility"}', call_id='call_2Qa2372bqTjLXyd9BSXQAmVc', name='search', type='function_call', id='fc_0ec6f9e72f7baad6006a3177b08f908192bb0571176c5bdc47', namespace=None, status='completed')

In [15]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered can I join admission late enrollment eligibility'}

In [16]:
call.name

'search'

In [17]:
results = search(**args)

In [18]:
result_json = json.dumps(results, indent=2)

In [19]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [20]:
messages.append(call)

In [21]:
messages.append(function_call_output)

In [22]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join admission late enrollment eligibility"}', call_id='call_2Qa2372bqTjLXyd9BSXQAmVc', name='search', type='function_call', id='fc_0ec6f9e72f7baad6006a3177b08f908192bb0571176c5bdc47', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_2Qa2372bqTjLXyd9BSXQAmVc',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a 

In [23]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [24]:
print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [25]:
response.usage

ResponseUsage(input_tokens=771, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=37, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=808)

In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [28]:
response = openai_client.responses.create(
    model="gpt-5.4-mini", input=messages, tools=[search_tool]
)

In [29]:
messages.extend(response.output)

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enroll discovered course can I join"}
function_call: search {"query":"new student enroll course join late registration FAQ"}


In [30]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll discovered course can I join"}', call_id='call_YO7vfX1mg7lKWj8BYBKPY5xR', name='search', type='function_call', id='fc_032cb93150dd4435006a3177b2a82481a08a4af031513778f1', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student enroll course join late registration FAQ"}', call_id='call_Mq0AwbQoeu8lAAM1SW57y

In [31]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini", input=messages, tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if not has_function_calls:
        break

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join"}
function_call: search {"query":"course enrollment join discovered late can I join"}
iteration #2...
ASSISTANT:
Yes, you can still join. The course materials are available, so you can start anytime.

One important caveat: if you want a certificate, you need to submit your project while submissions are still open. Certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help you with where to start, how the weekly workflow works, or how submissions/certificates work.


In [32]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model, input=messages, tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if not has_function_calls:
            break

    return last_answer

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

In [34]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered late enroll can I join course registration late joining FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.  

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.

If you’d like, I can also help you figure out how to start from where you are now.


In [35]:
result

'Yes — you can still join the course.  \n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.\n\nIf you’d like, I can also help you figure out how to start from where you are now.'

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening Queen's Gambit"}
iteration #2...
function_call: search {"query":"queen gambit course FAQ queen's gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any FAQ entry about “queen gambit.” If you meant the chess opening, that’s outside the course FAQ/logistics I’m able to answer here.

If you want, I can help with another course-related question. Are there other areas you want to explore?


In [37]:
from toyaikit.tools import Tools

In [38]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )